In [1]:
# Minimal reproducibility bootstrap for this notebook
import warnings
import random
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Bootstrap complete. Seed={SEED}")


Bootstrap complete. Seed=42


# Recruiting Pipeline Crew

## What We're Building

A four-agent recruiting workflow for structured candidate review:

| Agent | Role |
|-------|------|
| Resume Screener | Score resume evidence against job requirements |
| Question Generator | Create targeted interview questions from resume + JD |
| Candidate Summarizer | Write a concise hiring-manager brief |
| Bias-Check Reviewer | Review the pipeline for fairness risks and blind spots |

## Collaboration Flow

1. Resume Screener evaluates fit signals from the resume and JD.
2. Question Generator prepares probing interview questions from the same source material.
3. Candidate Summarizer combines screening and interview prep into one hiring brief.
4. Bias-Check Reviewer audits the combined process for fairness concerns.

This keeps role boundaries clear: evaluation, probing, synthesis, then fairness review.

## Stack
- CrewAI - multi-agent orchestration
- Ollama - local LLM endpoint


In [2]:
CREWAI_AVAILABLE = True

try:
    from crewai import Agent, Task, Crew, Process, LLM
except Exception:
    CREWAI_AVAILABLE = False
    Agent = Task = Crew = Process = LLM = None
    print("CrewAI is not installed. Install with: pip install crewai")

llm = None
if CREWAI_AVAILABLE:
    llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
    print(f"LLM ready: {llm.model}")
else:
    print("Running in demo mode without CrewAI runtime.")


CrewAI is not installed. Install with: pip install crewai
Running in demo mode without CrewAI runtime.


## Step 2 — Job Description & Candidate Data

In [3]:
job_description = """
POSITION: Senior Backend Engineer
COMPANY: CloudScale Inc. (Series B, 120 employees)
LOCATION: Remote (US time zones)
SALARY: $160K-$200K + equity

REQUIREMENTS:
- 5+ years backend development (Python or Go preferred)
- Experience with distributed systems (microservices, message queues)
- Strong SQL + at least one NoSQL database
- Cloud infrastructure (AWS or GCP)
- CI/CD pipelines and containerization (Docker, Kubernetes)
- API design (REST or gRPC)

NICE TO HAVE:
- Experience with event-driven architecture (Kafka, RabbitMQ)
- Performance optimization at scale (>10K rps)
- Open-source contributions
- Tech lead or mentorship experience

CULTURE:
- Small, senior team — high autonomy, low meetings
- Ship fast, iterate, own your domain
- Code review culture, strong testing expectations
"""

candidate_resume = """
NAME: Priya Sharma
EMAIL: priya.sharma@email.com
LOCATION: Austin, TX

SUMMARY:
Backend engineer with 6 years of experience building scalable data pipelines
and microservices. Currently at a fintech startup processing 50K+ transactions/sec.

EXPERIENCE:

Senior Backend Engineer — PayFlow (fintech startup, 45 employees)
Jan 2022 — Present (3 years)
- Designed event-driven payment processing pipeline using Kafka + Python
- Reduced API latency from 200ms to 45ms by optimizing PostgreSQL queries
- Built real-time fraud detection service handling 50K transactions/sec
- Migrated monolith to 12 microservices on Kubernetes (AWS EKS)
- Mentored 3 junior engineers; led weekly architecture review sessions
- On-call rotation lead; reduced P1 incidents by 60%

Backend Engineer — DataVault (data analytics, 200 employees)
Jun 2019 — Dec 2021 (2.5 years)
- Built ETL pipelines processing 2TB/day using Python + Apache Airflow
- Designed REST APIs serving 15 internal teams (FastAPI + PostgreSQL)
- Implemented Redis caching layer, reducing DB load by 40%
- Wrote comprehensive test suites (pytest), maintained 92% code coverage

Junior Developer — TechStart Agency
Aug 2018 — May 2019 (10 months)
- Built client websites and APIs using Django + PostgreSQL
- First exposure to Docker and CI/CD (GitHub Actions)

EDUCATION:
B.S. Computer Science — UT Austin (2018)

SKILLS:
Python, Go (learning), PostgreSQL, Redis, MongoDB, Kafka, Docker, Kubernetes,
AWS (EKS, RDS, S3, Lambda), FastAPI, Django, gRPC, pytest, GitHub Actions,
Terraform, Datadog

OPEN SOURCE:
- Contributor to Apache Airflow (3 merged PRs — DAG serialization fixes)
- Maintainer of `py-kafka-tools` (450 GitHub stars)
"""

print("Job description and candidate resume loaded")

Job description and candidate resume loaded


## Step 3 — Create Agents

In [4]:
if CREWAI_AVAILABLE:
    resume_screener = Agent(
        role="Resume Screener",
        goal="Score resume evidence against the JD using explicit criteria",
        backstory=(
            "You are a structured hiring reviewer who focuses on evidence quality,"
            " required skills, and honest fit assessment."
        ),
        llm=llm,
        verbose=True,
    )

    question_generator = Agent(
        role="Question Generator",
        goal="Generate targeted interview questions based on claimed experience and role needs",
        backstory=(
            "You design interview prompts that test depth, not buzzwords, and you tie"
            " every question to something visible in the candidate record."
        ),
        llm=llm,
        verbose=True,
    )

    candidate_summarizer = Agent(
        role="Candidate Summarizer",
        goal="Summarize candidate fit, uncertainties, and interview focus areas",
        backstory=(
            "You write short hiring briefs that help interviewers prepare without"
            " overstating certainty."
        ),
        llm=llm,
        verbose=True,
    )

    bias_checker = Agent(
        role="Bias-Check Reviewer",
        goal="Audit the recruiting pipeline for fairness issues and suggest corrections",
        backstory=(
            "You review candidate evaluation workflows for bias risks such as pedigree,"
            " affinity, recency, and confirmation bias."
        ),
        llm=llm,
        verbose=True,
    )
else:
    resume_screener = {"role": "Resume Screener"}
    question_generator = {"role": "Question Generator"}
    candidate_summarizer = {"role": "Candidate Summarizer"}
    bias_checker = {"role": "Bias-Check Reviewer"}

print("4 agents created: Resume Screener, Question Generator, Candidate Summarizer, Bias-Check Reviewer")


4 agents created: Resume Screener, Question Generator, Candidate Summarizer, Bias-Check Reviewer


## Step 4 — Create Tasks

**Parallelism**: `screen_task` and `question_task` both have `async_execution=True`.
They run concurrently since they both read the same inputs independently.
The `summary_task` uses `context=[screen_task, question_task]` to wait for both.

## Step 4 - Create Tasks

Parallelism is intentionally removed here to keep the recruiting workflow easier to inspect and audit step by step.


if CREWAI_AVAILABLE:
    screen_task = Task(
        description=f"""Evaluate this candidate against the job description.

JOB DESCRIPTION:
{job_description}

CANDIDATE RESUME:
{candidate_resume}

Return:
1. Match score and reasoning
2. Strengths evidenced in resume
3. Missing or uncertain areas
4. Interview risk flags""",
        expected_output="Structured resume screening with fit score and evidence.",
        agent=resume_screener,
    )

    question_task = Task(
        description=f"""Generate interview questions for this candidate using the JD and resume.

JOB DESCRIPTION:
{job_description}

CANDIDATE RESUME:
{candidate_resume}

Create:
1. Technical deep-dive questions
2. System/design questions
3. Behavioral questions
4. Clarifying questions for uncertain claims

For each, note what a strong answer should demonstrate.""",
        expected_output="Targeted interview question set with evaluation guidance.",
        agent=question_generator,
    )

    summary_task = Task(
        description="""Using the screening output and question plan, write a hiring-manager summary:
1. Candidate snapshot
2. Evidence-backed strengths
3. Top uncertainties to probe
4. Recommended interview focus
5. Preliminary recommendation with confidence level""",
        expected_output="Hiring manager brief with recommendation and focus areas.",
        agent=candidate_summarizer,
        context=[screen_task, question_task],
    )

    bias_task = Task(
        description="""Review the screening, question set, and candidate summary for bias risks.

Check for:
1. Pedigree bias
2. Affinity bias
3. Recency bias
4. Confirmation bias
5. Name/gender coded assumptions

For each issue found, propose a concrete correction.""",
        expected_output="Bias audit with issues found, fairness notes, and corrections.",
        agent=bias_checker,
        context=[screen_task, summary_task],
        markdown=True,
    )
else:
    screen_task = {"name": "Resume screening task"}
    question_task = {"name": "Question generation task"}
    summary_task = {"name": "Candidate summary task"}
    bias_task = {"name": "Bias review task"}

print("4 tasks created with flow: screening -> questions -> candidate summary -> bias review")


In [5]:
class _DemoTaskOutput:
    def __init__(self, name, raw):
        self.name = name
        self.raw = raw


class _DemoResult:
    def __init__(self):
        self.raw = "Bias review complete: candidate is promising, interview focus areas are defined, and fairness corrections are documented."
        self.tasks_output = [
            _DemoTaskOutput("Resume Screening", "Candidate meets core backend/distributed systems criteria with some domain uncertainty."),
            _DemoTaskOutput("Interview Questions", "Targeted technical and behavioral questions generated from resume evidence."),
            _DemoTaskOutput("Candidate Summary", "Hiring brief prepared with strengths, risks, and interview priorities."),
            _DemoTaskOutput("Bias Review", "Potential pedigree and recency bias flagged with mitigation guidance."),
        ]


if CREWAI_AVAILABLE:
    recruiting_crew = Crew(
        agents=[resume_screener, question_generator, candidate_summarizer, bias_checker],
        tasks=[screen_task, question_task, summary_task, bias_task],
        process=Process.sequential,
        verbose=True,
    )

    print("Recruiting pipeline crew assembled.")
    result = recruiting_crew.kickoff()
    print("\nBias audit complete.")
else:
    print("Demo mode: showing recruiting flow without live CrewAI kickoff.")
    result = _DemoResult()


Demo mode: showing recruiting flow without live CrewAI kickoff.


In [6]:
print("BIAS AUDIT REPORT (final output)")
print("=" * 60)
print(result.raw)


BIAS AUDIT REPORT (final output)
Bias review complete: candidate is promising, interview focus areas are defined, and fairness corrections are documented.


In [7]:
def preview(label, text):
    print("\n" + ("=" * 60))
    print(label)
    print("=" * 60)
    print(text)


preview("Resume Screening", result.tasks_output[0].raw)
preview("Interview Questions", result.tasks_output[1].raw)
preview("Candidate Summary", result.tasks_output[2].raw)
preview("Bias Review", result.tasks_output[3].raw)



Resume Screening
Candidate meets core backend/distributed systems criteria with some domain uncertainty.

Interview Questions
Targeted technical and behavioral questions generated from resume evidence.

Candidate Summary
Hiring brief prepared with strengths, risks, and interview priorities.

Bias Review
Potential pedigree and recency bias flagged with mitigation guidance.


## Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| Role specialization | Screening, interview design, summary writing, fairness review |
| Evidence handoff | Summary and bias review consume earlier outputs via `context=[...]` |
| Conservative decisioning | Outputs identify uncertainty rather than pretending certainty |
| Fairness review | Final stage checks process quality, not just candidate quality |

## Limitations

- This notebook should support human review, not replace hiring decisions.
- Resume text can be incomplete, exaggerated, or formatted inconsistently.
- Bias checks can flag obvious patterns but cannot guarantee fairness.
- Job fit signals from text alone are weaker than structured interviews and work samples.
- Any scoring should be treated as directional, not as a final ranking decision.

## Extensions

- Add work-sample review as another evidence source
- Integrate ATS systems for structured candidate ingestion
- Track interviewer calibration across candidates
- Export hiring brief and bias review into recruiter workflows


---

## 📚 CrewAI Series Summary (Notebooks 41-50)

| # | Project | Key CrewAI Concept |
|---|---------|-------------------|
| 41 | Startup Idea Validation | Agent, Task, Crew, Process basics |
| 42 | Content Creation | Sequential context passing (auto-chain) |
| 43 | Lead Generation | `context=[]` explicit task dependencies |
| 44 | Job Hunt Assistant | `allow_delegation=True` |
| 45 | Academic Research | `markdown=True` formatted output |
| 46 | Real Estate Analysis | Task `callback` functions |
| 47 | Product Launch | Large crews (5 agents), exec distillation |
| 48 | Competitive Intelligence | Threat categorization, BLUF memos |
| 49 | Customer Success | `output_file` auto-save results |
| 50 | Recruiting Pipeline | `async_execution` parallel tasks |